---
## Setup

In [ ]:
import torch
from pathlib import Path
from torchvision.datasets import CelebA
from torch import nn
from transformers import CLIPModel, CLIPProcessor
from torch.utils.tensorboard import SummaryWriter
import torch.nn.functional as F
import torchvision.transforms as transforms
from torchvision.transforms import InterpolationMode

/home/disi/deep_learning/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---
## Data

In [ ]:
def get_celeba(root, batch_size, download=False):
    """Create celeba dataloaders.

    Args:
        root: path to the stored celeba data
        batch_size: how many samples to process in parallel. GPU-dependent
        download: whether to download the dataset or not

    Returns:
        training, testing and validation data loaders
    """
    
    # Replicate CLIP's native image processing natively in PyTorch
    transform = transforms.Compose([
        transforms.Resize((224, 224), interpolation=InterpolationMode.BICUBIC),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=(0.48145466, 0.4578275, 0.40821073),
            std=(0.26862954, 0.26130258, 0.27577711)
        )
    ])

    training_data = CelebA(root=root, split="train", download=download, transform=transform)
    validation_data = CelebA(root=root, split="valid", download=download, transform=transform)
    test_data = CelebA(root=root, split="test", download=download, transform=transform)

    print(f"# of training samples: {len(training_data)}")
    print(f"# of validation samples: {len(validation_data)}")
    print(f"# of test samples: {len(test_data)}")

    train_loader = torch.utils.data.DataLoader(training_data, batch_size, shuffle=True, num_workers=4, pin_memory=True)
    val_loader = torch.utils.data.DataLoader(validation_data, int(batch_size/2), shuffle=False, num_workers=4, pin_memory=True)
    test_loader = torch.utils.data.DataLoader(test_data, batch_size, shuffle=False, num_workers=4, pin_memory=True)

    return train_loader, val_loader, test_loader

In [ ]:
"""
def get_celeba(root, batch_size, download = False):
  transform = transforms.ToTensor()

  training_data = CelebA(root=root, split="train", download=False, transform=transform)
  validation_data = CelebA(root=root, split="valid", download=False, transform=transform)
  test_data = CelebA(root=root, split="test", download=False, transform=transform)

  print(f"# of training samples: {len(training_data)}")
  print(f"# of validation samples: {len(validation_data)}")
  print(f"# of test samples: {len(test_data)}")

  train_loader = torch.utils.data.DataLoader(training_data, batch_size, shuffle=True)
  val_loader = torch.utils.data.DataLoader(validation_data, int(batch_size/2), shuffle=False)
  test_loader = torch.utils.data.DataLoader(test_data, batch_size, shuffle=False)

  return train_loader, val_loader, test_loader
"""

---
## Images and text features


In [ ]:
def get_image_features(clip_model, images):
    """Compute batch image features with the CLIP image encoder.
    
    Args:
        clip_model: Hugging Face CLIPModel
        images: batch images (already processed by dataloader)
        
    Returns:
        batch image features
    """
    # encode the images directly with CLIP
    with torch.no_grad():
        images_z_output = clip_model.get_image_features(pixel_values=images)

    images_z = images_z_output.pooler_output # the tensors

    return images_z

In [4]:
def get_normalized_image_features(images_features, device = "cuda"):
    """Retunr the image features normalized.

    Args:
        images_features: image features
        device: where to perform calculations

    Returns:
        batch image features normalized
    """
    images_z = images_features / images_features.norm(dim=-1, keepdim=True)

    return images_z

In [5]:
def encode_text_with_image(model, input_ids, img_tokens):
    """
    Injects an image token into CLIP text embeddings just before the End-of-Text (EOT) token
    assuming as text "a photo of".

    Args:
        model: a frozen Hugging Face CLIPModel
        input_ids: text token IDs, shape [batch_size, n_ctx] (typically n_ctx=77)
        img_tokens: image features to inject, shape [batch_size, hidden_size]

    Returns:
        Projected multimodal features, shape [batch_size, projection_dim]
    """
    model.eval()
    batch_size = input_ids.size(0)

    with torch.no_grad():
        # get the base token embeddings from CLIP
        token_embeds = model.text_model.embeddings.token_embedding(input_ids)

    # find the EOT token index (usually the largest, the one at the end)
    eot_indices = input_ids.argmax(dim=-1)

    # take the join index from the first item in the batch
    splice_idx = eot_indices[0].item()

    # insert the image tokens into the embeddings
    img_tokens = img_tokens.view(batch_size, 1, -1)
    spliced_embeds = torch.cat([
        token_embeds[:, :splice_idx],
        img_tokens,
        token_embeds[:, splice_idx:-1]
    ], dim=1)

    # create a runtime patch for the embedding layer's forward method
    original_embeddings_forward = model.text_model.embeddings.forward

    def patched_embeddings_forward(
        input_ids=None,
        position_ids=None,
        inputs_embeds=None
    ):
        # we intercept the call, drop input_ids, and force feed our custom
        # token embeddings
        return original_embeddings_forward(
            input_ids=None,
            position_ids=position_ids,
            inputs_embeds=spliced_embeds
        )

    try:
        # apply the runtime patch
        model.text_model.embeddings.forward = patched_embeddings_forward

        # run high-level text model safely
        outputs = model.text_model(input_ids=input_ids)
        last_hidden_state = outputs.last_hidden_state
    finally:
        # restore original method to avoid breaking downstream text tracking
        model.text_model.embeddings.forward = original_embeddings_forward

    # extract features at the shifted EOS token position
    # (+1 due to the injection)
    new_eot_indices = eot_indices + 1
    batch_indices = torch.arange(batch_size, device=input_ids.device)
    eot_features = last_hidden_state[batch_indices, new_eot_indices]

    projected_features = model.text_projection(eot_features)

    return projected_features

In [ ]:
def get_normalized_text_features(clip_model, token_features, text_inputs, device = "cuda"):
    """Compute text features with the CLIP text encoder.

    Args:
        clip_model: Hugging Face CLIPModel
        clip_processor: Hugging Face CLIPProcessor
        token_features: image features
        device: where to perform calculations

    Returns:
        text features
    """
    # extract input_ids
    input_ids = text_inputs["input_ids"].to(device)

    # repeat the tokenized row to match the batch size of token_features
    input_ids = input_ids.repeat(token_features.size(0), 1)  # Shape: [batch_size, 77]

    # features + pseudo language token
    text_features = encode_text_with_image(clip_model, input_ids, token_features)

    # normalize the features
    text_features = text_features / text_features.norm(dim=-1, keepdim=True)

    return text_features

In [ ]:
def a_photo_of_tokens(clip_processor, device = "cuda"):
    # tokenize text
    text_inputs = clip_processor(
        text="a photo of",
        padding="max_length",
        max_length=77,
        truncation=True,
        return_tensors="pt"
    )
    text_inputs = text_inputs.to(device)
    
    return text_inputs

---
## Model

In [7]:
class IM2TEXT(nn.Module):
    """pic2word model (MLP network).

    Attributes:
        fc_out: output layer
        layers: layers of the network
    """
    def __init__(self, embed_dim=512, middle_dim=512, output_dim=512, n_layer=2, dropout=0.1):
        """Initializes the model.

        Args:
            embed_dim: embedding dimension of images
            middle_dim: dimension of teh middle layer
            output_dim: dimension of the output layer
            n_layer: number of layers
            dropout: dropout rate
        """
        super().__init__()
        self.fc_out = nn.Linear(middle_dim, output_dim)
        layers = []
        dim = embed_dim
        for _ in range(n_layer):
            block = []
            block.append(nn.Linear(dim, middle_dim))
            block.append(nn.Dropout(dropout))
            block.append(nn.ReLU())
            dim = middle_dim
            layers.append(nn.Sequential(*block))
        self.layers = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor):
        """Forward method.

        Args:
            x: input tensor

        Returns:
            output tensor
        """
        for layer in self.layers:
            x = layer(x)
        return self.fc_out(x)

---
## Training

### Optimizer

In [8]:
def get_optimizer(img2text, wd = 0.02, lr = 1e-4):
    """Return the training (AdamW) optimizer of pic2word.

    Args:
        img2text: pic2word model
        wd: weight decay
        lr: learning rate

    Returns:
        optimizer
    """
    named_parameters = list(img2text.named_parameters())
    exclude = lambda n : "bn" in n or "ln" in n or "bias" in n or 'logit_scale' in n
    include = lambda n : not exclude(n)
    gain_or_bias_params = [p for n, p in named_parameters if exclude(n) and p.requires_grad]
    rest_params = [p for n, p in named_parameters if include(n) and p.requires_grad]

    optimizer = torch.optim.AdamW(
        [
            {"params": gain_or_bias_params, "weight_decay": 0.0},
            {"params": rest_params, "weight_decay": wd},
        ],
        lr=lr,
    )

    return optimizer

### Loss function

In [ ]:
def get_loss_img2text(clip_model, clip_processor, loss_img, loss_txt, images_features, token_features, text_inputs, device = "cuda"):
    """Get the training loss (contrastive loss) of pic2word

    Args:
        clip_model: Hugging Face CLIPModel
        clip_processor: Hugging Face CLIPProcessor
        loss_img: loss function (cross entropy loss) for images
        loss_txt: loss function (cross entropy loss) for text
        images_features: image features
        token_features: text features of the embedded pseudo-language token
        device: where to perform calculations

    Returns:
        image features, text features and loss
    """
    image_features = get_normalized_image_features(images_features)
    text_features = get_normalized_text_features(clip_model, clip_processor, token_features, text_inputs)

    logit_scale = clip_model.logit_scale.exp()
    logit_scale = logit_scale.mean()

    ground_truth = torch.arange(len(image_features)).long()
    ground_truth = ground_truth.to(device)

    logits_per_image = logit_scale * image_features @ text_features.t()
    loss_img_val = loss_img(logits_per_image, ground_truth)
    logits_per_text = logit_scale * text_features @ image_features.t()
    loss_txt_val = loss_txt(logits_per_text, ground_truth)
    total_loss = (loss_img_val + loss_txt_val) / 2

    return image_features, text_features, total_loss

### Training step

In [ ]:
def training_step(
    clip_model,
    clip_processor,
    img2text,
    training_data_loader,
    text_inputs,
    optimizer,
    loss_img,
    loss_txt,
    epoch,
    log_file,
    device="cuda",
    tb_writer=None
):
    """Training step of pic2word.

    Args:
        clip_model: Hugging Face CLIPModel
        clip_processor: Hugging Face CLIPProcessor
        img2text: pic2word model
        training_data_loader: training data loader
        optimizer: training optimizer (AdamW)
        loss_img: loss function (cross entropy loss) for images
        loss_txt: loss function (cross entropy loss) for text
        epoch: current epoch
        device: where to perform calculations
        tb_writer: tensorboard writer
    """
    num_batches_per_epoch = len(training_data_loader)

    # set the network to training mode
    img2text.train()
    # iterate over the training set
    for i, (images, _) in enumerate(training_data_loader):
        # store the actual step of the training
        step = num_batches_per_epoch * epoch + i

        # reset gradients
        optimizer.zero_grad()

        # move input images to cuda
        images = images.to(device)
        # extract CLIP features
        clip_image_features = get_image_features(
            clip_model,
            images
        )

        # forward pass: img2text processes the CLIP image features
        img_tokens = img2text(clip_image_features)

        # loss computation: pass raw images and img_tokens to the loss function
        image_features, text_features, loss = get_loss_img2text(
            clip_model,
            clip_processor,
            loss_img,
            loss_txt,
            clip_image_features,
            img_tokens,
            text_inputs,
            device
        )

        # backward pass (compute the gradients)
        loss.backward()

        # parameters update
        optimizer.step()

        # every 64 batches print training evolution
        if (i%64) == 0:
            num_samples = i * len(images)
            samples_per_epoch = len(training_data_loader.dataset)
            percent_complete = 100.0 * i / num_batches_per_epoch
            # print status of training
            msg = (
                f"Train Epoch: {epoch} [{num_samples}/{samples_per_epoch} ({percent_complete:.0f}%)]\t"
                f"Loss: {loss.item():.6f}"
                f"\tLR: {optimizer.param_groups[0]['lr']:5f}\tlogit_scale {clip_model.logit_scale.data:.3f}"
            )
            print(msg)
            # append status to the log file
            with open(log_file, "a") as f:
                f.write(msg + "\n")
                f.flush()

        # every 32 batches store loss updates (also scale and lr, that for now
        # are static) on the tensor board
        if (i%32) == 0:
            timestep = epoch * num_batches_per_epoch + i
            log_data = {
                "loss": loss.item(),
                "scale":  clip_model.logit_scale.data.item(),
                "lr": optimizer.param_groups[0]["lr"],
                "cossim": cosine_similarity_mean(text_features, image_features)
            }

            for name, val in log_data.items():
                name = "train/" + name
                if tb_writer is not None:
                    tb_writer.add_scalar(name, val, timestep)
                    

---
## Testing

### Metrics

In [ ]:
def cosine_similarity_mean(text_pseudo_z, image_z):
    """Computes cosine similarity mean.

    Args:
        text_pseudo_z: text features modified with pseudo-language tokens
        image_z: normalized image features

    Returns:
        cosine similarity mean
    """
    return F.cosine_similarity(text_pseudo_z, image_z, dim=-1).mean().cpu()

In [12]:
def cosine_similarity(text_pseudo_z, image_z):
    """Computes cosine similarity.

    Args:
        text_pseudo_z: text features modified with pseudo-language tokens
        image_z: normalized image features

    Returns:
        cosine similarity
    """
    return F.cosine_similarity(text_pseudo_z.unsqueeze(1), image_z.unsqueeze(0), dim=-1).cpu()

### Testing step

In [ ]:
def val_step(
    clip_model,
    clip_processor,
    img2text,
    val_data_loader,
    text_inputs,
    loss_img,
    loss_txt,
    epoch,
    log_file,
    tb_writer=None,
    device="cuda",
):
    num_batches_per_epoch = len(val_data_loader)

    # set the network to validation/testing mode
    img2text.eval()

    # iterate over the training set
    with torch.no_grad():
      for i, (images, _) in enumerate(val_data_loader):
          # store the actual step of the training
          step = num_batches_per_epoch * epoch + i
          # move input images to cuda
          images = images.to(device)
          # extract CLIP features from the raw images
          # extract CLIP features (images are already preprocessed and on device)
          clip_image_features = get_image_features(
              clip_model,
              images
          )
          # obtain pseudo-language tokens
          img_tokens = img2text(clip_image_features)
          # loss computation: pass raw images and img_tokens to the loss function
          image_features, text_features, loss = get_loss_img2text(
              clip_model,
              clip_processor,
              loss_img,
              loss_txt,
              clip_image_features,
              img_tokens,
              text_inputs,
              device
          )

          # every 8 batches print validation evolution
          if (i%8) == 0:
              num_samples = i * len(images)
              samples_per_epoch = len(val_data_loader.dataset)
              percent_complete = 100.0 * i / num_batches_per_epoch
              # print status of training
              msg = (
                  f"Validation Epoch: {epoch} [{num_samples}/{samples_per_epoch} ({percent_complete:.0f}%)]\t"
              )
              print(msg)
              # append status to the log file
              with open(log_file, "a") as f:
                  f.write(msg + "\n")
                  f.flush()
                  
              timestep = epoch * num_batches_per_epoch + i
              log_data = {
                  "loss": loss.item(),
                  "cossim": cosine_similarity_mean(text_features, image_features)
              }

              for name, val in log_data.items():
                  name = "val/" + name
                  if tb_writer is not None:
                      tb_writer.add_scalar(name, val, timestep)


---
## Main

In [ ]:
def main(
    clip_model,
    clip_processor,
    exp_name="exp1",
    root=Path("/content/datasets"),
    batch_size=128,     # how many samples to process in parallel. GPU-dependent
    device="cuda",    # Where to perform calculations
    wd = 0.02,
    lr = 1e-4,
    epochs=8          # How many times to iterate over the entire train set
):
    # create a logger for the experiment
    writer = SummaryWriter(log_dir=f"runs/{exp_name}")
    log_file = f"runs/{exp_name}.log"

    # get dataloaders
    train_loader, val_loader, _ = get_celeba(root, batch_size)
    text_inputs = a_photo_of_tokens(clip_processor)
    # frozen CLIP model
    clip_model = clip_model.cuda().eval()
    for param in clip_model.parameters():
        param.requires_grad = False

    # instantiate the network and move it to the chosen device (GPU)
    img2text = IM2TEXT().to(device)

    # "print" the network to view all the modules, so its architecture
    print(img2text)

    # instantiate the optimizer
    optimizer = get_optimizer(img2text, wd, lr)

    # create loss functions and move them to the GPU
    loss_img = nn.CrossEntropyLoss().to(device)
    loss_txt = nn.CrossEntropyLoss().to(device)

    print("Training:")
    # for each epoch, train the network and then compute evaluation results
    for e in range(epochs):
        training_step(
            clip_model,
            clip_processor,
            img2text,
            train_loader,
            text_inputs,
            optimizer,
            loss_img,
            loss_txt,
            e,
            log_file,
            device="cuda",
            tb_writer=writer
        )
        val_step(clip_model, clip_processor, img2text, val_loader, text_inputs, loss_img, loss_txt, e, log_file, writer)

    # closes the logger
    writer.close()
    # store the trained model
    torch.save(img2text.state_dict(), f"runs/model_{exp_name}.pth")

    return img2text

## Burn the GPU

In [15]:
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

Loading weights: 100%|████████████████████| 398/398 [00:00<00:00, 40880.96it/s]


In [28]:
net = main(clip_model, processor, exp_name="exp1_real", root=Path("/home/disi/deep_learning/DeepLearning2026/data/celeba"), batch_size = 256)

# of training samples: 162770
# of validation samples: 19867
# of test samples: 19962
IM2TEXT(
  (fc_out): Linear(in_features=512, out_features=512, bias=True)
  (layers): Sequential(
    (0): Sequential(
      (0): Linear(in_features=512, out_features=512, bias=True)
      (1): Dropout(p=0.1, inplace=False)
      (2): ReLU()
    )
    (1): Sequential(
      (0): Linear(in_features=512, out_features=512, bias=True)
      (1): Dropout(p=0.1, inplace=False)
      (2): ReLU()
    )
  )
)
Training:
Train Epoch: 0 [0/162770 (0%)]	Loss: 5.606314	LR: 0.000100	logit_scale 4.605


KeyboardInterrupt: 